# V2_03 — Stage 1: Stacked Ensemble

5-fold out-of-fold (OOF) stacking with Ridge meta-learner over 3 base models:
XGBoost, CatBoost, LightGBM.

Each fold trains 3 models on 80% of training data, predicts on the held-out 20%.
The meta-learner learns optimal weights from the OOF predictions.

**Runtime:** T4 GPU + High-RAM | ~6-8 hrs | ~12-16 CU

**Prerequisite:** Gold parquets in `My Drive/AllowanceMap/V2/gold/`. Run after V2_01.

In [ ]:
# ── Cell 1: Environment Setup ──────────────────────────────────────────────
import os, subprocess, sys

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
    'mlflow>=2.12', 'catboost>=1.2', 'lightgbm>=4.3', 'databricks-sdk>=0.20'])

from google.colab import drive, userdata
drive.mount('/content/drive')

DRIVE_ROOT = '/content/drive/MyDrive/AllowanceMap/V2'
GOLD_DIR   = f'{DRIVE_ROOT}/gold'
ARTIFACTS  = f'{DRIVE_ROOT}/v2_artifacts'
os.makedirs(f'{ARTIFACTS}/models', exist_ok=True)
os.makedirs(f'{ARTIFACTS}/predictions', exist_ok=True)
os.makedirs(f'{ARTIFACTS}/plots', exist_ok=True)

os.environ['DATABRICKS_HOST']  = 'https://dbc-d709cbb6-fe84.cloud.databricks.com'
os.environ['DATABRICKS_TOKEN'] = userdata.get('DATABRICKS_TOKEN')

import mlflow, requests
mlflow.set_tracking_uri('databricks')
resp = requests.get(
    f"{os.environ['DATABRICKS_HOST']}/api/2.0/preview/scim/v2/Me",
    headers={'Authorization': f"Bearer {os.environ['DATABRICKS_TOKEN']}"},
    timeout=10,
)
resp.raise_for_status()
USER_HOME = f"/Users/{resp.json()['userName']}"
mlflow.set_experiment(f'{USER_HOME}/medicare_models')
print(f'MLflow: {USER_HOME}/medicare_models')

In [ ]:
# ── Cell 2: Constants & Utilities ──────────────────────────────────────────
import glob, gc, time, json, pickle
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split, KFold
from sklearn.linear_model import RidgeCV
import xgboost as xgb
from catboost import CatBoostRegressor, Pool
import lightgbm as lgb
import matplotlib.pyplot as plt

TARGET = 'Avg_Mdcr_Alowd_Amt'

FEATURES = [
    'Rndrng_Prvdr_Type_idx', 'Rndrng_Prvdr_State_Abrvtn_idx',
    'HCPCS_Cd_idx', 'hcpcs_bucket', 'place_of_srvc_flag',
    'Bene_Avg_Risk_Scre', 'log_srvcs', 'log_benes',
    'Avg_Sbmtd_Chrg', 'srvcs_per_bene',
    'specialty_bucket', 'pos_bucket', 'hcpcs_target_enc',
]

CAT_FEATURE_NAMES = [
    'Rndrng_Prvdr_Type_idx', 'Rndrng_Prvdr_State_Abrvtn_idx',
    'HCPCS_Cd_idx', 'hcpcs_bucket', 'place_of_srvc_flag',
    'specialty_bucket', 'pos_bucket',
]

N_FOLDS = 5
XGB_ROUNDS = 1000
CB_ITERS = 3000
LGB_ROUNDS = 1000
EARLY_STOP = 50

def get_cat_indices(features):
    return [i for i, f in enumerate(features) if f in CAT_FEATURE_NAMES]

def make_pool(X_df, y, features, cat_indices):
    """Create CatBoost Pool from DataFrame with proper categorical handling."""
    X = X_df[features].copy()
    for idx in cat_indices:
        col = features[idx]
        X[col] = X[col].astype(int)
    return Pool(X, label=y, cat_features=cat_indices, feature_names=features)

def compute_metrics(y_true_log, y_pred_log):
    yt = np.expm1(y_true_log)
    yp = np.expm1(y_pred_log)
    return {
        'mae': mean_absolute_error(yt, yp),
        'rmse': np.sqrt(mean_squared_error(yt, yp)),
        'r2': r2_score(yt, yp),
    }

In [ ]:
# ── Cell 3: Load Full Dataset → save splits to local SSD ──────────────────
print('Loading all gold parquets...')
t0 = time.time()

load_cols = list(dict.fromkeys(FEATURES + [TARGET]))   # no 'year' needed
parts = []
for f in sorted(glob.glob(os.path.join(GOLD_DIR, '*.parquet'))):
    avail = set(pq.read_schema(f).names)
    cols = [c for c in load_cols if c in avail]
    df = pd.read_parquet(f, columns=cols).dropna(subset=FEATURES + [TARGET])
    parts.append(df)

df_all = pd.concat(parts, ignore_index=True)
del parts; gc.collect()
print(f'Loaded {len(df_all):,} rows in {time.time()-t0:.0f}s')

# log1p target
df_all[TARGET] = np.log1p(df_all[TARGET].astype('float64'))

# Same 80/20 split as V2_01 for comparable test set
idx_train, idx_test = train_test_split(np.arange(len(df_all)), test_size=0.2, random_state=42)
n_train = len(idx_train)
n_test  = len(idx_test)

df_all.iloc[idx_train].to_parquet('/content/train_split.parquet', index=False)
df_all.iloc[idx_test].to_parquet('/content/test_split.parquet', index=False)
del df_all; gc.collect()

print(f'Train (for CV): {n_train:,} | Test (holdout): {n_test:,}')
print('Saved splits to /content/train_split.parquet & /content/test_split.parquet')

In [ ]:
# ── Cell 4: Hyperparameters ────────────────────────────────────────────────
XGB_PARAMS = {
    'objective': 'reg:squarederror', 'learning_rate': 0.05, 'max_depth': 6,
    'subsample': 0.8, 'colsample_bytree': 0.8, 'tree_method': 'hist',
    'device': 'cuda', 'seed': 42, 'max_bin': 256,
}

CB_PARAMS = {
    'loss_function': 'RMSE', 'learning_rate': 0.05, 'depth': 6,
    'bootstrap_type': 'MVS', 'subsample': 0.8,
    'task_type': 'GPU',
    'random_seed': 42, 'verbose': 0, 'allow_writing_files': False,
    'iterations': CB_ITERS,
}

LGB_PARAMS = {
    'objective': 'regression', 'metric': 'rmse', 'learning_rate': 0.05,
    'num_leaves': 63, 'max_depth': -1, 'subsample': 0.8,
    'colsample_bytree': 0.8, 'min_child_samples': 20,
    'device': 'cpu', 'num_threads': -1,
    'seed': 42, 'verbose': -1,
    'boosting_type': 'gbdt', 'data_sample_strategy': 'goss',
}

cat_idx = get_cat_indices(FEATURES)
cat_cols = [FEATURES[i] for i in cat_idx]
print(f'Cat indices: {cat_idx}')
print(f'Ready for {N_FOLDS}-fold CV with 3 base learners')

## 5-Fold Out-of-Fold Training

For each fold:
1. Train XGBoost, CatBoost, LightGBM on fold training set
2. Predict on fold validation set → OOF predictions
3. Predict on holdout test set → averaged across folds

In [ ]:
# ── Cell 5: OOF Training Loop ──────────────────────────────────────────────
# Reload splits from local SSD — keep both numpy (XGB/LGB) and DataFrame (CatBoost)
# Peak ~30 GB: ~20 GB numpy train + ~10 GB DataFrame train → fits 51 GB High-RAM
df_train_full = pd.read_parquet('/content/train_split.parquet')
df_test       = pd.read_parquet('/content/test_split.parquet')

y_train_full = df_train_full[TARGET].values
y_test       = df_test[TARGET].values

# Numpy arrays for XGB and LGB (float32 to save memory)
X_train_full_np = df_train_full[FEATURES].astype('float32').values
X_test_np       = df_test[FEATURES].astype('float32').values

# DataFrame slices for CatBoost Pool construction
X_df_train_full = df_train_full[FEATURES].copy()
X_df_test       = df_test[FEATURES].copy()
del df_train_full, df_test; gc.collect()

kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=42)

# OOF predictions: each row gets a prediction from the fold where it was in validation
oof_preds = np.zeros((n_train, 3))   # columns: [XGB, CatBoost, LightGBM]
# Test predictions: averaged across folds
test_preds = np.zeros((n_test, 3))

total_t0 = time.time()

for fold, (tr_idx, val_idx) in enumerate(kf.split(X_train_full_np)):
    print(f'\n{"="*60}')
    print(f'FOLD {fold+1}/{N_FOLDS}  |  train: {len(tr_idx):,}  |  val: {len(val_idx):,}')
    print('='*60)

    # ── XGBoost ────────────────────────────────────────────────────────
    print(f'  [XGB] Training...')
    t0 = time.time()
    dtrain_f = xgb.DMatrix(X_train_full_np[tr_idx], label=y_train_full[tr_idx], feature_names=FEATURES)
    dval_f   = xgb.DMatrix(X_train_full_np[val_idx], label=y_train_full[val_idx], feature_names=FEATURES)
    dtest_f  = xgb.DMatrix(X_test_np, feature_names=FEATURES)

    booster = xgb.train(
        XGB_PARAMS, dtrain_f, num_boost_round=XGB_ROUNDS,
        evals=[(dval_f, 'val')], early_stopping_rounds=EARLY_STOP, verbose_eval=0,
    )
    oof_preds[val_idx, 0] = booster.predict(dval_f)
    test_preds[:, 0] += booster.predict(dtest_f) / N_FOLDS
    m = compute_metrics(y_train_full[val_idx], oof_preds[val_idx, 0])
    print(f'  [XGB] R²={m["r2"]:.4f}  MAE=${m["mae"]:.2f}  ({time.time()-t0:.0f}s, {booster.best_iteration} rounds)')
    del dtrain_f, dval_f, dtest_f, booster; gc.collect()

    # ── CatBoost ──────────────────────────────────────────────────────
    print(f'  [CB]  Training...')
    t0 = time.time()
    pool_tr  = make_pool(X_df_train_full.iloc[tr_idx].reset_index(drop=True),
                         y_train_full[tr_idx], FEATURES, cat_idx)
    pool_val = make_pool(X_df_train_full.iloc[val_idx].reset_index(drop=True),
                         y_train_full[val_idx], FEATURES, cat_idx)
    pool_te  = make_pool(X_df_test, None, FEATURES, cat_idx)

    cb = CatBoostRegressor(**CB_PARAMS)
    cb.fit(pool_tr, eval_set=pool_val, early_stopping_rounds=EARLY_STOP, use_best_model=True)
    oof_preds[val_idx, 1] = cb.predict(pool_val)
    test_preds[:, 1] += cb.predict(pool_te) / N_FOLDS
    m = compute_metrics(y_train_full[val_idx], oof_preds[val_idx, 1])
    print(f'  [CB]  R²={m["r2"]:.4f}  MAE=${m["mae"]:.2f}  ({time.time()-t0:.0f}s, {cb.best_iteration_} iters)')
    del pool_tr, pool_val, pool_te, cb; gc.collect()

    # ── LightGBM ──────────────────────────────────────────────────────
    print(f'  [LGB] Training...')
    t0 = time.time()
    ds_tr = lgb.Dataset(X_train_full_np[tr_idx], label=y_train_full[tr_idx],
                        feature_name=FEATURES, categorical_feature=cat_cols, free_raw_data=True)
    ds_val = lgb.Dataset(X_train_full_np[val_idx], label=y_train_full[val_idx],
                         feature_name=FEATURES, categorical_feature=cat_cols,
                         reference=ds_tr, free_raw_data=True)

    lgb_b = lgb.train(
        LGB_PARAMS, ds_tr, num_boost_round=LGB_ROUNDS,
        valid_sets=[ds_val], callbacks=[lgb.early_stopping(EARLY_STOP), lgb.log_evaluation(0)],
    )
    oof_preds[val_idx, 2] = lgb_b.predict(X_train_full_np[val_idx])
    test_preds[:, 2] += lgb_b.predict(X_test_np) / N_FOLDS
    m = compute_metrics(y_train_full[val_idx], oof_preds[val_idx, 2])
    print(f'  [LGB] R²={m["r2"]:.4f}  MAE=${m["mae"]:.2f}  ({time.time()-t0:.0f}s, {lgb_b.best_iteration} rounds)')
    del ds_tr, ds_val, lgb_b; gc.collect()

total_time = time.time() - total_t0
print(f'\nAll {N_FOLDS} folds complete in {total_time/3600:.1f} hours')

# Save OOF + test predictions and y arrays to local SSD, then free large arrays
np.save('/content/oof_preds.npy', oof_preds)
np.save('/content/test_preds.npy', test_preds)
np.save('/content/y_train_full.npy', y_train_full)
np.save('/content/y_test.npy', y_test)
del X_train_full_np, X_test_np, X_df_train_full, X_df_test, y_train_full, y_test; gc.collect()
print('Predictions saved to /content/*.npy — training data freed')

In [ ]:
# ── Cell 6: OOF Metrics per Base Learner ───────────────────────────────────
# Reload predictions from local SSD
oof_preds    = np.load('/content/oof_preds.npy')
test_preds   = np.load('/content/test_preds.npy')
y_train_full = np.load('/content/y_train_full.npy')
y_test       = np.load('/content/y_test.npy')

print('\nOOF Metrics (computed on full training set, each row predicted once):')
base_names = ['XGBoost', 'CatBoost', 'LightGBM']
oof_results = {}

for i, name in enumerate(base_names):
    m = compute_metrics(y_train_full, oof_preds[:, i])
    oof_results[name] = m
    print(f'  {name:12s}  MAE=${m["mae"]:8.2f}  RMSE=${m["rmse"]:8.2f}  R²={m["r2"]:.4f}')

## Ridge Meta-Learner

Learns optimal weights to combine the 3 base model predictions.
Trained on OOF predictions (no data leakage — each prediction was made on held-out data).

In [ ]:
# ── Cell 7: Train Meta-Learner ─────────────────────────────────────────────
meta = RidgeCV(alphas=[0.001, 0.01, 0.1, 1.0, 10.0, 100.0])
meta.fit(oof_preds, y_train_full)

print(f'Ridge alpha: {meta.alpha_}')
print(f'Ridge coefficients: {dict(zip(base_names, meta.coef_))}')
print(f'Ridge intercept: {meta.intercept_:.6f}')

# Ensemble prediction on test set
y_pred_ensemble = meta.predict(test_preds)
ensemble_metrics = compute_metrics(y_test, y_pred_ensemble)

print(f'\nEnsemble Test Metrics:')
print(f'  MAE=${ensemble_metrics["mae"]:.2f}  RMSE=${ensemble_metrics["rmse"]:.2f}  R²={ensemble_metrics["r2"]:.4f}')

# Individual base learner test metrics for comparison
print(f'\nBase Learner Test Metrics (averaged across folds):')
for i, name in enumerate(base_names):
    m = compute_metrics(y_test, test_preds[:, i])
    print(f'  {name:12s}  MAE=${m["mae"]:8.2f}  RMSE=${m["rmse"]:8.2f}  R²={m["r2"]:.4f}')

best_single_r2 = max(compute_metrics(y_test, test_preds[:, i])['r2'] for i in range(3))
ensemble_lift = ensemble_metrics['r2'] - best_single_r2
print(f'\nEnsemble R² lift over best single model: {ensemble_lift:+.4f}')

In [ ]:
# ── Cell 8: Log Ensemble to MLflow ─────────────────────────────────────────
with mlflow.start_run(run_name='ensemble_stack_v2_colab'):
    mlflow.log_params({
        'model': 'StackedEnsemble',
        'base_learners': 'XGBoost+CatBoost+LightGBM',
        'meta_learner': 'RidgeCV',
        'n_folds': N_FOLDS,
        'meta_alpha': float(meta.alpha_),
        'meta_coef_xgb': float(meta.coef_[0]),
        'meta_coef_catboost': float(meta.coef_[1]),
        'meta_coef_lgbm': float(meta.coef_[2]),
        'meta_intercept': float(meta.intercept_),
        'xgb_rounds': XGB_ROUNDS,
        'cb_iters': CB_ITERS,
        'lgb_rounds': LGB_ROUNDS,
        'early_stopping': EARLY_STOP,
        'source': 'colab',
        'strategy': 'stacking_5fold_oof',
        'target_transform': 'log1p',
        'sample_frac': 1.0,
        'split_strategy': 'random',
        'n_features': len(FEATURES),
        'ablation_avg_submitted_charge': False,
        'version': 'v2',
        'training_time_sec': int(total_time),
        'n_train': n_train,
        'n_test': n_test,
    })

    mlflow.log_metrics({
        'test_mae': ensemble_metrics['mae'],
        'test_rmse': ensemble_metrics['rmse'],
        'test_r2': ensemble_metrics['r2'],
        'ensemble_lift_over_best_single': ensemble_lift,
    })

    # Log base learner test metrics
    for i, name in enumerate(base_names):
        m = compute_metrics(y_test, test_preds[:, i])
        mlflow.log_metrics({
            f'base_{name.lower()}_test_mae': m['mae'],
            f'base_{name.lower()}_test_rmse': m['rmse'],
            f'base_{name.lower()}_test_r2': m['r2'],
        })

    # Log meta-learner as artifact
    meta_path = f'{ARTIFACTS}/models/ensemble_ridge.pkl'
    with open(meta_path, 'wb') as f:
        pickle.dump(meta, f)
    mlflow.log_artifact(meta_path, artifact_path='ensemble')
    mlflow.sklearn.log_model(meta, artifact_path='ridge_meta')

print('Ensemble logged to MLflow.')

In [ ]:
# ── Cell 9: Save OOF Predictions & Plots ──────────────────────────────────
# Save OOF predictions for future analysis
oof_df = pd.DataFrame(oof_preds, columns=['xgb_oof', 'catboost_oof', 'lgbm_oof'])
oof_df['y_true'] = y_train_full
oof_df.to_parquet(f'{ARTIFACTS}/predictions/ensemble_oof.parquet', index=False)
print(f'OOF predictions saved: {len(oof_df):,} rows')

# Save test predictions
test_df = pd.DataFrame(test_preds, columns=['xgb_test', 'catboost_test', 'lgbm_test'])
test_df['ensemble_pred'] = y_pred_ensemble
test_df['y_true'] = y_test
test_df.to_parquet(f'{ARTIFACTS}/predictions/ensemble_test.parquet', index=False)

# Ensemble weight visualization
fig, ax = plt.subplots(figsize=(8, 4))
colors = ['#2196F3', '#4CAF50', '#FF9800']
bars = ax.bar(base_names, meta.coef_, color=colors, edgecolor='black', linewidth=0.5)
ax.axhline(y=0, color='black', linewidth=0.5)
ax.set_ylabel('Ridge Coefficient (Weight)')
ax.set_title(f'Ensemble Stacking Weights (alpha={meta.alpha_})')
for bar, val in zip(bars, meta.coef_):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{val:.3f}', ha='center', va='bottom', fontsize=11)
plt.tight_layout()
plt.savefig(f'{ARTIFACTS}/plots/ensemble_weights.png', dpi=150)
plt.show()
print(f'Plot saved to {ARTIFACTS}/plots/ensemble_weights.png')

In [ ]:
# ── Cell 10: Final Summary ─────────────────────────────────────────────────
print('\n' + '='*70)
print('V2_03 STACKED ENSEMBLE COMPLETE')
print('='*70)
print(f'Dataset: {n_train + n_test:,} total rows (no sampling)')
print(f'Strategy: {N_FOLDS}-fold OOF stacking with RidgeCV meta-learner')
print(f'Base learners: XGBoost, CatBoost, LightGBM')
print(f'Training time: {total_time/3600:.1f} hours')
print()
print(f'Ridge weights: XGB={meta.coef_[0]:.3f}  CB={meta.coef_[1]:.3f}  LGB={meta.coef_[2]:.3f}')
print()
print('Test Set Results:')
for i, name in enumerate(base_names):
    m = compute_metrics(y_test, test_preds[:, i])
    print(f'  {name:12s}  MAE=${m["mae"]:8.2f}  RMSE=${m["rmse"]:8.2f}  R²={m["r2"]:.4f}')
print(f'  {"ENSEMBLE":12s}  MAE=${ensemble_metrics["mae"]:8.2f}  RMSE=${ensemble_metrics["rmse"]:8.2f}  R²={ensemble_metrics["r2"]:.4f}')
print(f'\n  Lift: {ensemble_lift:+.4f} R² over best single model')
print()
print('V1 baselines:')
print('  RF (V1)       MAE=$12.04   RMSE=$22.73   R²=0.8843  (0.3 sample, batch)')
print('  XGB (V1)      MAE=$11.83   RMSE=$21.18   R²=0.8331  (0.3 sample, batch)')